# Working with prompts

In [1]:
# setting the environment variables, the keys
import sys
import os

sys.path.insert(0, os.path.abspath('..'))

from config import set_environment
# for the keys - as explained early in chapter 2
set_environment()

In [2]:
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import GoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser

# llm = GoogleGenerativeAI(model="gemini-1.5-pro")
llm = GoogleGenerativeAI(model="gemini-1.5-flash-latest")

# First chain generates a story
story_prompt = PromptTemplate.from_template("Write a short story about {topic}")
story_chain = story_prompt | llm | StrOutputParser()

# Second chain analyzes the story
analysis_prompt = PromptTemplate.from_template(
    "Analyze the following story's mood:\n{story}"
)
analysis_chain = analysis_prompt | llm | StrOutputParser()

# Combine chains
story_with_analysis = story_chain | analysis_chain

# Run the combined chain
story_analysis = story_with_analysis.invoke({"topic": "a rainy day"})
print("\nAnalysis:", story_analysis)


Analysis: The story's mood is initially one of **gloomy introspection and loneliness**, mirroring Elara's feelings about the overwhelming storm.  The descriptions of the rain – "relentless percussion," "mournful howl," "watery apocalypse" – establish a somber and somewhat ominous atmosphere.  Elara's boredom and isolation further contribute to this feeling.

However, the mood dramatically shifts as Elara discovers the photographs and the yellow umbrella.  The discovery introduces elements of **mystery, intrigue, and ultimately, warmth and connection**. The initial loneliness gives way to a sense of wonder and belonging as she connects with her family history.  The "quiet warmth" that blossoms in the attic contrasts sharply with the stormy exterior, creating a powerful juxtaposition.

The final mood is one of **hopeful nostalgia and quiet joy**.  The storm continues, but its impact is lessened by Elara's newfound understanding and sense of continuity across generations. The "whispered 

Now preserve context through the chain (show the story too)

In [5]:
from langchain_core.runnables import RunnablePassthrough

# Using RunnablePassthrough.assign to preserve data
enhanced_chain = RunnablePassthrough.assign(
    story=story_chain # Add 'story' key with generated content
).assign(
    analysis=analysis_chain # Add 'analysis' key with analysis of the story
)

# Execute the chain

result = enhanced_chain.invoke({"topic": "a rainy day"})
print(result.keys())

dict_keys(['topic', 'story', 'analysis'])


In [4]:
from operator import itemgetter
# Alternative approach using dictionary construction

manual_chain = (
    RunnablePassthrough() | # Pass through input
    {
        "story": story_chain, # Add story result
        "topic": itemgetter("topic") # Preserve original topic
    } |
    RunnablePassthrough().assign( # Add analysis based on story
        analysis=analysis_chain
    )
)

result = manual_chain.invoke({"topic": "a rainy day"})
print(result) # output: {"story": "story content...", "topic": "topic content...", "analysis": "analysis content..."}

{'story': 'The rain hammered against the attic window, a relentless tattoo against the aged glass.  Eleven-year-old Elara, perched on a dusty trunk, watched the deluge transform the world outside into a blurry watercolour painting.  The attic, usually a stuffy, forgotten space, felt strangely comforting today.  The air hung heavy with the scent of damp wood and old paper, a scent that usually repelled her, but today felt oddly soothing.\n\nHer grandmother, Nana Elsie, was downstairs, humming a tuneless melody as she kneaded dough for her famous apple pie.  The rhythmic thud of the rolling pin was a counterpoint to the rain\'s rhythm, a comforting duet against the grey backdrop of the storm.\n\nElara traced a finger across a faded photograph tucked into a leather-bound book.  It was a picture of a young woman, her smile bright even in the sepia tones, standing beneath a similar downpour.  Nana Elsie had told her it was her mother, a woman who loved rainy days as much as Elara did.\n\n"S

We can simplify this with dictionary conversion using a LCEL shorthand:

In [7]:
# Simplified dictionary construction
simple_dict_chain_corrected = story_chain | {
    "story": RunnablePassthrough(), # Pass the story output as 'story'
    "analysis": analysis_chain
}

# analysis_chain will receive {'story': 'the actual story content'} as expected
result_corrected = simple_dict_chain_corrected.invoke({"topic": "a rainy day"})
print(result_corrected)

dict_keys(['story', 'analysis'])


## LLMs and prompts

In [5]:
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import GoogleGenerativeAI

# Create a template with variables
template = """
Summarize this text in one sentence:

{text}
"""
# llm = GoogleGenerativeAI(model="gemini-1.5-pro")
llm = GoogleGenerativeAI(model="gemini-2.5-flash")

prompt = PromptTemplate.from_template(template)

# Format the prompt with actual values
formatted_prompt = prompt.format(text="Some long story about AI...")

# Use with any LLM, such as the one created in the LLM section
result = llm.invoke(formatted_prompt)
print(result)

This is a long story about AI.


## Chat models and prompts

In [6]:
from langchain_core.prompts import ChatPromptTemplate
# from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

template = ChatPromptTemplate.from_messages([
    ("system", "You are an English to French translator."),
    ("user", "Translate this to French: {text}")
])

# chat = ChatOpenAI()
chat = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
formatted_messages = template.format_messages(text="Hello, how are you?")
result = chat.invoke(formatted_messages)
print(result.content)

Bonjour, comment allez-vous ?
